# CoalitionGNN — Preprocessing

Ham veri (`data/raw/`) → Kanonik tablolar (`data/canonical/`) → Graf snapshot'ları (`data/snapshots/`).

**Ön koşul:** `01_download_data_v2.ipynb` çalıştırılmış, tüm kaynaklar OK durumda.

**Kapsam dönemi:** 1946-2016 (COW state system system2016.csv sınırına göre).

**Bu notebook'un yapmadığı:** Koalisyon örnekleri üretimi, etiket hesaplama. Onlar `03_coalition_samples.ipynb`'da olacak.

**Sonuçlar bir adım-adım rapora yazılır** (her aşama: kaç satır okundu, kaç ülke eşleşti, ne kadar veri kaybedildi).

## 0. Hazırlık

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
RAW_DIR = os.path.join(PROJECT_DIR, 'data/raw')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
os.makedirs(CANON_DIR, exist_ok=True)
os.makedirs(SNAP_DIR, exist_ok=True)

# Proje sabitleri
START_YEAR = 1946
END_YEAR = 2016
SMOOTH_WINDOW = 5  # 5 yıllık kayan pencere (manifest v3)
TRAIN_END = 1999   # z-skor normalizasyonu için train/test sınırı

print(f'Ham:       {RAW_DIR}')
print(f'Kanonik:   {CANON_DIR}')
print(f'Snapshot:  {SNAP_DIR}')
print(f'Dönem:     {START_YEAR}-{END_YEAR}')

In [ ]:
!pip install -q pandas pyarrow torch torch_geometric pycountry numpy tqdm

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from pathlib import Path
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Adım-adım rapor toplayıcısı
stages = []

def log_stage(name, status, details, n_rows=None, n_countries=None):
    """Bir preprocessing aşamasının sonucunu kaydet."""
    entry = {
        'stage': name,
        'status': status,  # OK / WARN / FAIL
        'details': details,
        'n_rows': n_rows,
        'n_countries': n_countries,
    }
    stages.append(entry)
    icon = {'OK': '✅', 'WARN': '⚠️', 'FAIL': '❌'}.get(status, '?')
    suffix = ''
    if n_rows is not None:
        suffix += f' [{n_rows:,} satır'
        if n_countries is not None:
            suffix += f', {n_countries} ülke'
        suffix += ']'
    print(f'{icon} {name}: {details}{suffix}')

def find_file(raw_subdir, pattern):
    """Ham veri dizini altında bir dosya ara. İlk eşleşeni döner."""
    full_dir = os.path.join(RAW_DIR, raw_subdir)
    matches = list(Path(full_dir).rglob(pattern))
    if not matches:
        raise FileNotFoundError(f'{pattern} bulunamadı: {full_dir}')
    return str(matches[0])

In [ ]:
# DİYAGNOSTİK: Ham veri dizinindeki tüm dosyaları listele
# Hangi dosyaların geldiğini, ne isimlerde olduğunu burada görüyoruz.
# find_file desenleri buna göre ayarlandı.

print('=== Ham veri dizini içeriği ===\n')
for subdir in sorted(os.listdir(RAW_DIR)):
    full = os.path.join(RAW_DIR, subdir)
    if not os.path.isdir(full):
        continue
    print(f'📁 {subdir}/')
    for root, dirs, files in os.walk(full):
        rel = os.path.relpath(root, full)
        prefix = '   ' if rel == '.' else f'   └ {rel}/'
        for f in sorted(files):
            size_kb = os.path.getsize(os.path.join(root, f)) / 1024
            print(f'{prefix} {f}  ({size_kb:.0f} KB)')
    print()

## 1. Ülke Kodu Çıpası — `country_master`

COW state system veri seti, hangi ülkenin hangi yıllarda 'var' olduğunu söyler.
Bu, sonraki tüm birleştirmelerin referansı.

**Önemli:** SSCB (365) → Rusya (365, aynı kod ama farklı dönem), Yugoslavya (345) → Sırbistan (345),
Çekoslovakya (315) → Çek (316) + Slovak (317). Bu geçişler COW kodlamasında zaten doğru kodlanmış.

def find_col(df, candidates, required=True):
    """Veri çerçevesinde aday isimlerden ilk eşleşeni bul (büyük/küçük harf duyarsız).
    
    Veri kaynaklarının sürüm farkları yüzünden sütun isimleri değişebilir;
    bu fonksiyon olası varyasyonları sırayla dener.
    """
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    if required:
        raise KeyError(
            f"Aşağıdaki sütunların hiçbiri bulunamadı: {candidates}\n"
            f"Mevcut sütunlar: {list(df.columns)}"
        )
    return None


In [ ]:
# COW state system membership — system2016.csv zaten yıllık panel formatında.
# Sütunlar: stateabb, ccode, year, version → her ülke-yıl çifti için bir satır.

try:
    sys_path = find_file('cow/cow_state_system', '*system*.csv')
    sys_df = pd.read_csv(sys_path)
    sys_df.columns = sys_df.columns.str.lower()
    
    print(f'Dosya: {os.path.basename(sys_path)}')
    print(f'Sütunlar: {list(sys_df.columns)}')
    print(f'\nİlk 3 satır:')
    print(sys_df.head(3))
    
    log_stage('1a_state_system', 'OK', f'okundu: {os.path.basename(sys_path)}',
              n_rows=len(sys_df), n_countries=sys_df['ccode'].nunique())
except Exception as e:
    log_stage('1a_state_system', 'FAIL', str(e))
    raise

In [ ]:
# system2016.csv ZATEN yıllık panel formatında — sadece dönem aralığına filtreliyoruz.
# (Önceki sürümlerde 'dönem' formatı vardı, styear-endyear sütunlarıyla. Bu sürüm farklı.)

country_master = sys_df[
    (sys_df['year'] >= START_YEAR) & (sys_df['year'] <= END_YEAR)
].copy()

country_master = country_master.rename(columns={
    'ccode': 'cow_code',
    'stateabb': 'state_name',
})[['year', 'cow_code', 'state_name']].drop_duplicates(['year', 'cow_code'])

country_master['cow_code'] = country_master['cow_code'].astype(int)
country_master['year'] = country_master['year'].astype(int)

country_master.to_parquet(os.path.join(CANON_DIR, 'country_master.parquet'))

log_stage('1b_country_master', 'OK',
          f'panel hazır, {START_YEAR}-{END_YEAR}',
          n_rows=len(country_master),
          n_countries=country_master['cow_code'].nunique())

# Sanity check: yıl başına ülke sayısı
by_year = country_master.groupby('year').size()
print(f'\nYıl başına ülke sayısı (sanity):')
for sample_year in [START_YEAR, 1960, 1990, 2000, END_YEAR]:
    if sample_year in by_year.index:
        print(f'  {sample_year}: {by_year.loc[sample_year]} ülke')

## 2. Düğüm Özellikleri — `nodes`

NMC'den kapasite endeksleri, Polity'den rejim tipi, V-Dem'den demokrasi alt-endeksleri.
Üçünü ülke-yıl panelinde birleştir.

In [ ]:
# --- NMC: kapasite ---
try:
    nmc_path = find_file('cow/cow_nmc', 'NMC*.csv')
    nmc_df = pd.read_csv(nmc_path, low_memory=False, encoding='latin-1')
    nmc_df.columns = nmc_df.columns.str.lower()
    
    print(f'NMC sütunları: {list(nmc_df.columns)}')
    
    # COW kodu sütunu
    cow_col = find_col(nmc_df, ['ccode', 'cowcode', 'cow_code'])
    if cow_col != 'cow_code':
        nmc_df = nmc_df.rename(columns={cow_col: 'cow_code'})
    
    # -9 = missing kodu, NaN'a çevir
    for col in ['milex', 'milper', 'irst', 'pec', 'tpop', 'upop', 'cinc']:
        if col in nmc_df.columns:
            nmc_df.loc[nmc_df[col] == -9, col] = np.nan
    
    nmc_df = nmc_df[(nmc_df['year'] >= START_YEAR) & (nmc_df['year'] <= END_YEAR)]
    
    log_stage('2a_nmc', 'OK', f'okundu: {os.path.basename(nmc_path)}',
              n_rows=len(nmc_df), n_countries=nmc_df['cow_code'].nunique())
except Exception as e:
    log_stage('2a_nmc', 'FAIL', str(e))
    nmc_df = None

In [ ]:
# --- Polity5: rejim ---
try:
    polity_path = find_file('polity', '*.xls*')
    polity_df = pd.read_excel(polity_path)
    polity_df.columns = polity_df.columns.str.lower()
    
    # Polity'de COW kodu zaten var: 'ccode'
    polity_df = polity_df.rename(columns={'ccode': 'cow_code'})
    
    # polity2 = sürekli demokrasi skoru, -10 ile +10 arası
    # -66, -77, -88 = özel kodlar (interregnum, transition, vb.) → NaN
    if 'polity2' in polity_df.columns:
        polity_df.loc[polity_df['polity2'].isin([-66, -77, -88]), 'polity2'] = np.nan
    
    polity_df = polity_df[['cow_code', 'year', 'polity2']].dropna(subset=['cow_code', 'year'])
    polity_df['cow_code'] = polity_df['cow_code'].astype(int)
    polity_df['year'] = polity_df['year'].astype(int)
    polity_df = polity_df[(polity_df['year'] >= START_YEAR) & (polity_df['year'] <= END_YEAR)]
    polity_df = polity_df.drop_duplicates(['cow_code', 'year'])
    
    log_stage('2b_polity', 'OK', f'okundu, polity2 skoru çekildi',
              n_rows=len(polity_df), n_countries=polity_df['cow_code'].nunique())
except Exception as e:
    log_stage('2b_polity', 'WARN', f'Polity okuması başarısız: {str(e)[:80]} — devam ediyoruz')
    polity_df = None

In [ ]:
# --- V-Dem: demokrasi alt-endeksleri + GDP per capita ---
# V-Dem'in country_id'si COW'a birebir uymaz; v2cowcode sütunu kullan
try:
    vdem_path = find_file('vdem', 'V-Dem-CY*.csv')
    
    # V-Dem 500+ sütun, hepsini okumak gereksiz — sadece istediklerimizi al
    wanted_cols = [
        'country_id', 'country_name', 'year',
        'COWcode',         # COW kodu (büyük/küçük harf varyasyonu olabilir)
        'v2x_polyarchy',   # elektoral demokrasi endeksi (0-1)
        'v2x_libdem',      # liberal demokrasi endeksi
        'v2x_partipdem',   # katılımcı demokrasi
        'e_gdppc',         # GDP per capita (Maddison, dolar)
        'e_pop',           # toplam nüfus
    ]
    
    # Önce sütun başlıklarını gör
    header = pd.read_csv(vdem_path, nrows=0)
    available = [c for c in wanted_cols if c in header.columns]
    # COWcode küçük/büyük harf varyasyonunu yakala
    cow_col = next((c for c in header.columns if c.lower() == 'cowcode'), None)
    if cow_col and cow_col not in available:
        available.append(cow_col)
    
    vdem_df = pd.read_csv(vdem_path, usecols=available, low_memory=False)
    
    if cow_col:
        vdem_df = vdem_df.rename(columns={cow_col: 'cow_code'})
    
    vdem_df = vdem_df.dropna(subset=['cow_code', 'year'])
    vdem_df['cow_code'] = vdem_df['cow_code'].astype(int)
    vdem_df['year'] = vdem_df['year'].astype(int)
    vdem_df = vdem_df[(vdem_df['year'] >= START_YEAR) & (vdem_df['year'] <= END_YEAR)]
    vdem_df = vdem_df.drop_duplicates(['cow_code', 'year'])
    
    # country_name'i analiz için tutmaya gerek yok
    drop_cols = [c for c in ['country_id', 'country_name'] if c in vdem_df.columns]
    vdem_df = vdem_df.drop(columns=drop_cols)
    
    log_stage('2c_vdem', 'OK', f'okundu, {len(vdem_df.columns)-2} özellik sütunu',
              n_rows=len(vdem_df), n_countries=vdem_df['cow_code'].nunique())
except Exception as e:
    log_stage('2c_vdem', 'WARN', f'V-Dem okuması başarısız: {str(e)[:80]} — devam ediyoruz')
    vdem_df = None

In [ ]:
# Düğüm özelliklerini country_master üzerine sol-birleştir
nodes = country_master.copy()

if nmc_df is not None:
    nmc_cols = ['cow_code', 'year', 'cinc', 'milex', 'milper', 'tpop']
    nmc_cols = [c for c in nmc_cols if c in nmc_df.columns]
    nodes = nodes.merge(nmc_df[nmc_cols], on=['cow_code', 'year'], how='left')

if polity_df is not None:
    nodes = nodes.merge(polity_df, on=['cow_code', 'year'], how='left')

if vdem_df is not None:
    nodes = nodes.merge(vdem_df, on=['cow_code', 'year'], how='left')

# Forward-fill: rejim/demokrasi skoru bir yıl eksikse, önceki yılın değerini kullan
# limit=3: en fazla 3 yıllık boşluk doldurulur, daha uzun boşluklar NaN kalır
ffill_cols = [c for c in ['polity2', 'v2x_polyarchy', 'v2x_libdem', 'v2x_partipdem'] if c in nodes.columns]
if ffill_cols:
    nodes = nodes.sort_values(['cow_code', 'year'])
    nodes[ffill_cols] = nodes.groupby('cow_code')[ffill_cols].ffill(limit=3)

# GDP/nüfus için: log dönüşüm + lineer interpolasyon (zaman içinde monoton hareket eder)
for col in ['e_gdppc', 'e_pop', 'tpop', 'milex', 'milper']:
    if col in nodes.columns:
        nodes[col] = nodes.groupby('cow_code')[col].transform(
            lambda s: s.interpolate(method='linear', limit=5, limit_direction='both')
        )
        nodes[f'log_{col}'] = np.log1p(nodes[col].clip(lower=0))

nodes.to_parquet(os.path.join(CANON_DIR, 'nodes.parquet'))

# Eksiklik raporu
missing_pct = (nodes.isna().sum() / len(nodes) * 100).round(1)
missing_summary = missing_pct[missing_pct > 0].to_dict()

log_stage('2d_nodes_merged', 'OK',
          f'{len(nodes.columns)} sütun (eksik: {missing_summary})',
          n_rows=len(nodes),
          n_countries=nodes['cow_code'].nunique())

print('\nDüğüm özelliği sütunları:', list(nodes.columns))

## 3. Kenar Tabloları

Her ilişki tipi için aynı şema: `(year, cow_i, cow_j, weight)`.
Yönsüz kenarlar (ittifak, çatışma) için `cow_i < cow_j` kuralıyla tekilleştiriyoruz.

### 3a. Ticaret

**Birincil kaynak:** COW Trade 4.0 (Dyadic_COW_4.0.csv) — zaten COW kodlu, dönüşüm gereksiz.
**İkincil kaynak (varsa):** IMF DOTS/IMTS — daha güncel ama ISO→COW dönüşümü gerekli.

COW Trade 4.0 kapsamı: 1870-2014, ikili ticaret akışları (ihracat + ithalat, USD milyon).

In [ ]:
import pycountry

# Öncelikle COW Trade 4.0'ı dene (zaten COW kodlu, dönüşüm gereksiz)
cow_trade_used = False

try:
    dyadic_path = find_file('cow/cow_trade', '*Dyadic*.csv')
    cow_trade_df = pd.read_csv(dyadic_path, low_memory=False)
    cow_trade_df.columns = cow_trade_df.columns.str.lower()
    
    print(f'COW Trade 4.0 sütunları: {list(cow_trade_df.columns)}')
    print(f'İlk satır:')
    print(cow_trade_df.head(2))
    
    # COW Trade sütunları: ccode1, ccode2, year, flow1 (1→2 export), flow2 (2→1 export)
    # -9 = missing
    cow_trade_df = cow_trade_df.rename(columns={'ccode1': 'cow_i', 'ccode2': 'cow_j'})
    cow_trade_df = cow_trade_df[(cow_trade_df['year'] >= START_YEAR) & (cow_trade_df['year'] <= END_YEAR)]
    
    # flow1 = ccode1'in ccode2'ye ihracatı, flow2 = ccode2'nin ccode1'e ihracatı
    # Her iki yönü topla: toplam ikili ticaret hacmi
    for col in ['flow1', 'flow2']:
        if col in cow_trade_df.columns:
            cow_trade_df.loc[cow_trade_df[col] == -9, col] = np.nan
    
    # Toplam akış: iki yönlü ihracat toplamı
    cow_trade_df['total_flow'] = cow_trade_df[['flow1', 'flow2']].sum(axis=1, skipna=True)
    cow_trade_df = cow_trade_df.dropna(subset=['total_flow'])
    cow_trade_df = cow_trade_df[cow_trade_df['total_flow'] > 0]
    
    # Yönsüzleştir (COW Trade zaten dyadic, i<j garantisi yok)
    cow_trade_df['c1'] = cow_trade_df[['cow_i', 'cow_j']].min(axis=1)
    cow_trade_df['c2'] = cow_trade_df[['cow_i', 'cow_j']].max(axis=1)
    
    trade_undirected = (
        cow_trade_df.groupby(['year', 'c1', 'c2'])['total_flow'].sum()
        .reset_index()
        .rename(columns={'c1': 'cow_i', 'c2': 'cow_j', 'total_flow': 'flow_usd'})
    )
    trade_undirected = trade_undirected[trade_undirected['cow_i'] != trade_undirected['cow_j']]
    
    cow_trade_used = True
    log_stage('3a_trade_read', 'OK',
              f'COW Trade 4.0 (birincil kaynak)',
              n_rows=len(trade_undirected))
              
except Exception as e:
    log_stage('3a_trade_read', 'WARN', f'COW Trade 4.0 okunamadı: {str(e)[:80]}')

# COW Trade başarısız olursa IMF DOTS'a fallback
if not cow_trade_used:
    try:
        exp_df = pd.read_parquet(os.path.join(RAW_DIR, 'imf_dots', 'dots_exports.parquet'))
        imp_df = pd.read_parquet(os.path.join(RAW_DIR, 'imf_dots', 'dots_imports.parquet'))
        log_stage('3a_trade_read', 'OK',
                  f'IMF DOTS (fallback kaynak)',
                  n_rows=len(exp_df) + len(imp_df))
    except Exception as e:
        log_stage('3a_trade_read', 'FAIL',
                  f'Hiçbir ticaret kaynağı okunamadı: {str(e)[:80]}')
        exp_df = imp_df = None

In [ ]:
# ISO-3 → COW dönüşüm tablosu
# Kapsamlı hardcoded tablo: COW state abbreviation → ISO-3 alpha
# Kaynak: correlatesofwar.org state system membership + ISO 3166-1 alpha-3

# Tam COW → ISO-3 eşlemesi (COW abbreviation → ISO-3 alpha)
COW_TO_ISO3 = {
    'USA': 'USA', 'CAN': 'CAN', 'BHM': 'BHS', 'CUB': 'CUB', 'HAI': 'HTI',
    'DOM': 'DOM', 'JAM': 'JAM', 'TRI': 'TTO', 'BAR': 'BRB', 'DMA': 'DMA',
    'GRN': 'GRD', 'SLU': 'LCA', 'SVG': 'VCT', 'AAB': 'ATG', 'SKN': 'KNA',
    'MEX': 'MEX', 'BLZ': 'BLZ', 'GUA': 'GTM', 'HON': 'HND', 'SAL': 'SLV',
    'NIC': 'NIC', 'COS': 'CRI', 'PAN': 'PAN', 'COL': 'COL', 'VEN': 'VEN',
    'GUY': 'GUY', 'SUR': 'SUR', 'ECU': 'ECU', 'PER': 'PER', 'BRA': 'BRA',
    'BOL': 'BOL', 'PAR': 'PRY', 'CHL': 'CHL', 'ARG': 'ARG', 'URU': 'URY',
    'UKG': 'GBR', 'IRE': 'IRL', 'NTH': 'NLD', 'BEL': 'BEL', 'LUX': 'LUX',
    'FRN': 'FRA', 'MNC': 'MCO', 'LIE': 'LIE', 'SWZ': 'CHE', 'SPN': 'ESP',
    'AND': 'AND', 'POR': 'PRT', 'GMY': 'DEU', 'GFR': 'DEU', 'GDR': 'DDR',
    'POL': 'POL', 'AUS': 'AUT', 'HUN': 'HUN', 'CZR': 'CZE', 'CZE': 'CZE',
    'SLO': 'SVK', 'ITA': 'ITA', 'SNM': 'SMR', 'MLT': 'MLT', 'ALB': 'ALB',
    'MNG': 'MNE', 'MKD': 'MKD', 'CRO': 'HRV', 'YUG': 'YUG', 'SRB': 'SRB',
    'BOS': 'BIH', 'KOS': 'XKX', 'SLV': 'SVN', 'GRC': 'GRC', 'CYP': 'CYP',
    'BUL': 'BGR', 'MLD': 'MDA', 'ROM': 'ROU', 'RUS': 'RUS', 'USR': 'RUS',
    'EST': 'EST', 'LAT': 'LVA', 'LIT': 'LTU', 'UKR': 'UKR', 'BLR': 'BLR',
    'ARM': 'ARM', 'GRG': 'GEO', 'AZE': 'AZE', 'FIN': 'FIN', 'SWD': 'SWE',
    'NOR': 'NOR', 'DEN': 'DNK', 'ICE': 'ISL',
    'MOR': 'MAR', 'ALG': 'DZA', 'TUN': 'TUN', 'LIB': 'LBY', 'SUD': 'SDN',
    'SSD': 'SSD', 'IRN': 'IRN', 'TUR': 'TUR', 'IRQ': 'IRQ', 'EGY': 'EGY',
    'SYR': 'SYR', 'LEB': 'LBN', 'JOR': 'JOR', 'ISR': 'ISR', 'SAU': 'SAU',
    'YEM': 'YEM', 'YAR': 'YEM', 'YPR': 'YEM', 'KUW': 'KWT', 'BAH': 'BHR',
    'QAT': 'QAT', 'UAE': 'ARE', 'OMA': 'OMN', 'AFG': 'AFG',
    'TKM': 'TKM', 'TAJ': 'TJK', 'KYR': 'KGZ', 'UZB': 'UZB', 'KZK': 'KAZ',
    'CHN': 'CHN', 'MON': 'MNG', 'TAW': 'TWN', 'ROK': 'KOR', 'PRK': 'PRK',
    'JPN': 'JPN', 'IND': 'IND', 'BHU': 'BTN', 'PAK': 'PAK', 'BNG': 'BGD',
    'MYA': 'MMR', 'SRI': 'LKA', 'MAD': 'MDV', 'NEP': 'NPL',
    'THI': 'THA', 'CAM': 'KHM', 'LAO': 'LAO', 'DRV': 'VNM', 'RVN': 'VNM',
    'VNM': 'VNM', 'MAL': 'MYS', 'SIN': 'SGP', 'BRU': 'BRN', 'PHI': 'PHL',
    'INS': 'IDN', 'ETM': 'TLS', 'AUL': 'AUS', 'PNG': 'PNG', 'NEW': 'NZL',
    'FIJ': 'FJI', 'SOL': 'SLB', 'VAN': 'VUT', 'WSM': 'WSM', 'TON': 'TON',
    'NAU': 'NRU', 'MSI': 'MHL', 'PAL': 'PLW', 'FSM': 'FSM', 'KIR': 'KIR',
    'TUV': 'TUV',
    'SEN': 'SEN', 'GAM': 'GMB', 'MLI': 'MLI', 'GUI': 'GIN', 'SIE': 'SLE',
    'LBR': 'LBR', 'CDI': 'CIV', 'GHA': 'GHA', 'TOG': 'TGO', 'BEN': 'BEN',
    'BFO': 'BFA', 'NIR': 'NER', 'CHA': 'TCD', 'NIG': 'NGA', 'CAO': 'CMR',
    'CEN': 'CAF', 'EQG': 'GNQ', 'GAB': 'GAB', 'CON': 'COG', 'DRC': 'COD',
    'ZAI': 'COD', 'UGA': 'UGA', 'KEN': 'KEN', 'TAZ': 'TZA', 'ZAN': 'TZA',
    'BUI': 'BDI', 'RWA': 'RWA', 'SOM': 'SOM', 'DJI': 'DJI', 'ETH': 'ETH',
    'ERI': 'ERI', 'ANG': 'AGO', 'MOZ': 'MOZ', 'ZAM': 'ZMB', 'ZIM': 'ZWE',
    'MAW': 'MWI', 'SAF': 'ZAF', 'NAM': 'NAM', 'LES': 'LSO', 'BOT': 'BWA',
    'SWA': 'SWZ', 'MAS': 'MUS', 'COM': 'COM', 'MAG': 'MDG', 'SEY': 'SYC',
    'CAP': 'CPV', 'SAO': 'STP', 'GNB': 'GNB',
}

def build_iso3_to_cow():
    """ISO-3 → COW eşleşmesi. Kapsamlı hardcoded tablo + pycountry fallback."""
    cow_lookup = (
        country_master.sort_values('year', ascending=False)
        .drop_duplicates('cow_code')[['cow_code', 'state_name']]
    )
    name_to_cow = dict(zip(cow_lookup['state_name'], cow_lookup['cow_code']))
    
    # Ters çevir: ISO-3 → COW kodu
    iso_to_cow = {}
    for cow_abbr, iso3 in COW_TO_ISO3.items():
        if cow_abbr in name_to_cow:
            # Aynı ISO-3'e birden fazla COW kodu gidebilir (YEM, VNM, DEU...)
            # En güncel olanı tut (zaten sort_values descending ile geldi)
            if iso3 not in iso_to_cow:
                iso_to_cow[iso3] = name_to_cow[cow_abbr]
    
    # pycountry fallback: kalan COW abbreviation'lar ISO-3'e mi uyuyor?
    for abbr, code in name_to_cow.items():
        if isinstance(abbr, str) and len(abbr) == 3 and abbr.isalpha() and abbr.upper() == abbr:
            try:
                pycountry.countries.lookup(abbr)
                if abbr not in iso_to_cow:
                    iso_to_cow[abbr] = code
            except LookupError:
                pass
    
    return iso_to_cow

iso3_to_cow = build_iso3_to_cow()
print(f'ISO-3 → COW eşleşmesi: {len(iso3_to_cow)} ülke')
print(f'Örnek: {dict(list(iso3_to_cow.items())[:10])}')

In [ ]:
# DOTS'tan kenar tablosu kur (sadece IMF DOTS kullanıldıysa)
# COW Trade kullanıldıysa bu hücre atlanır — trade_undirected zaten hazır.

if not cow_trade_used and exp_df is not None and imp_df is not None:
    def normalize_trade(df, flow_label):
        df = df.copy()
        df.columns = [c.lower() for c in df.columns]
        
        reporter_col = next((c for c in df.columns if c in ['reporter', 'ref_area']), None)
        partner_col = next((c for c in df.columns if 'counterpart' in c or 'partner' in c), None)
        time_col = next((c for c in df.columns if c in ['time_period', 'year'] or 'time' in c), None)
        value_col = next((c for c in df.columns if c in ['obs_value', 'value']), None)
        
        if not all([reporter_col, partner_col, time_col, value_col]):
            raise ValueError(
                f'Beklenen sütunlar eksik.\n'
                f'  reporter: {reporter_col}, partner: {partner_col}, time: {time_col}, value: {value_col}\n'
                f'  Mevcut sütunlar: {df.columns.tolist()}'
            )
        
        df = df.rename(columns={reporter_col: 'reporter', partner_col: 'partner',
                                time_col: 'year', value_col: 'value'})
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df['flow'] = flow_label
        return df[['reporter', 'partner', 'year', 'value', 'flow']].dropna(subset=['value'])
    
    exp_clean = normalize_trade(exp_df, 'export')
    imp_clean = normalize_trade(imp_df, 'import')
    trade = pd.concat([exp_clean, imp_clean], ignore_index=True)
    
    # ISO-2 → ISO-3 dönüşüm (yeni API ISO-2 kullanıyor)
    def iso2_to_iso3(code):
        if not isinstance(code, str):
            return None
        if len(code) == 3:
            return code
        if len(code) == 2:
            try:
                return pycountry.countries.get(alpha_2=code.upper()).alpha_3
            except (LookupError, AttributeError):
                return None
        return None
    
    sample_code = trade['reporter'].dropna().iloc[0] if len(trade) > 0 else ''
    if len(str(sample_code)) == 2:
        print('  IMF verisi ISO-2 formatında, ISO-3\'e dönüştürülüyor...')
        trade['reporter'] = trade['reporter'].apply(iso2_to_iso3)
        trade['partner'] = trade['partner'].apply(iso2_to_iso3)
    
    # ISO-3 → COW dönüşüm
    trade['cow_i'] = trade['reporter'].map(iso3_to_cow)
    trade['cow_j'] = trade['partner'].map(iso3_to_cow)
    
    n_before = len(trade)
    trade = trade.dropna(subset=['cow_i', 'cow_j', 'year', 'value'])
    n_after = len(trade)
    
    trade['cow_i'] = trade['cow_i'].astype(int)
    trade['cow_j'] = trade['cow_j'].astype(int)
    trade = trade[(trade['year'] >= START_YEAR) & (trade['year'] <= END_YEAR)]
    trade = trade[trade['cow_i'] != trade['cow_j']]
    
    # Simetrikleştir
    trade['c1'] = trade[['cow_i', 'cow_j']].min(axis=1)
    trade['c2'] = trade[['cow_i', 'cow_j']].max(axis=1)
    trade_undirected = (
        trade.groupby(['year', 'c1', 'c2'])['value'].sum()
        .reset_index()
        .rename(columns={'c1': 'cow_i', 'c2': 'cow_j', 'value': 'flow_usd'})
    )
    
    log_stage('3a_trade_cow_mapping', 'OK',
              f'COW eşleşmesi sonrası %{100*n_after/n_before:.1f} kaldı',
              n_rows=len(trade_undirected))

elif not cow_trade_used:
    print('⚠️  Ticaret verisi yok — kenar tablosu atlanacak')
    trade_undirected = None

In [ ]:
# Log dönüşüm + z-skor (trade_undirected hangi kaynaktan geldiyse)

if trade_undirected is not None and len(trade_undirected) > 0:
    # Log dönüşüm
    trade_undirected['log_flow'] = np.log1p(trade_undirected['flow_usd'].clip(lower=0))
    
    # Z-skor — sadece train dönemi (<=TRAIN_END) istatistikleriyle, zamansal sızıntıyı önlemek için
    train_mask = trade_undirected['year'] <= TRAIN_END
    mean = trade_undirected.loc[train_mask, 'log_flow'].mean()
    std = trade_undirected.loc[train_mask, 'log_flow'].std()
    if std < 1e-6 or pd.isna(std):
        std = 1.0
    trade_undirected['weight'] = (trade_undirected['log_flow'] - mean) / std
    
    edges_trade = trade_undirected[['year', 'cow_i', 'cow_j', 'weight']].copy()
    edges_trade.to_parquet(os.path.join(CANON_DIR, 'edges_trade.parquet'))
    
    log_stage('3a_trade_edges', 'OK',
              f'simetrik kenarlar, log+z-skor (train mean/std)',
              n_rows=len(edges_trade))
else:
    log_stage('3a_trade_edges', 'WARN', 'ticaret verisi yok, kenar tablosu üretilmedi')

### 3b. İttifaklar (COW Formal Alliances)

Yönsüz binary kenar. ATOP yerine COW alliances v4.1 kullanıyoruz çünkü COW kodu zaten oturmuş.

In [ ]:
try:
    # COW alliance v4.1: alliance_v4.1_by_dyad_yearly.csv en kullanışlısı
    ally_path = find_file('cow/cow_alliances', '*dyad*yearly*.csv')
    ally_df = pd.read_csv(ally_path, low_memory=False)
    ally_df.columns = ally_df.columns.str.lower()
    
    # Sütunlar: ccode1, ccode2, year, defense, neutrality, nonaggression, entente
    ally_df = ally_df.rename(columns={'ccode1': 'cow_i', 'ccode2': 'cow_j'})
    ally_df = ally_df[(ally_df['year'] >= START_YEAR) & (ally_df['year'] <= END_YEAR)]
    
    # Bizim için kritik tip: savunma + saldırmazlık (en koalisyonel)
    ally_df['has_alliance'] = (
        ally_df.get('defense', 0).astype(int) |
        ally_df.get('nonaggression', 0).astype(int)
    )
    ally_df = ally_df[ally_df['has_alliance'] == 1]
    
    # Yönsüzleştir
    ally_df['c1'] = ally_df[['cow_i', 'cow_j']].min(axis=1)
    ally_df['c2'] = ally_df[['cow_i', 'cow_j']].max(axis=1)
    edges_ally = (
        ally_df[['year', 'c1', 'c2']]
        .drop_duplicates()
        .rename(columns={'c1': 'cow_i', 'c2': 'cow_j'})
    )
    edges_ally['weight'] = 1.0
    edges_ally = edges_ally[edges_ally['cow_i'] != edges_ally['cow_j']]
    
    edges_ally.to_parquet(os.path.join(CANON_DIR, 'edges_ally.parquet'))
    log_stage('3b_ally_edges', 'OK',
              f'defense+nonaggression, binary',
              n_rows=len(edges_ally))
except Exception as e:
    log_stage('3b_ally_edges', 'FAIL', str(e))

### 3c. BM Oylama Yakınlığı (Voeten ideal points)

İdeal point dosyasından her yıl için her ülke çifti arasında mesafe = |ideal_i - ideal_j|.
Düşük mesafe = yüksek yakınlık. k-NN yaklaşımı: her ülke için en yakın K komşu kenar olur.
Kenar ağırlığı = 1/(1+mesafe), [0,1] arası.

In [ ]:
try:
    # IdealpointestimatesAll dosyası — yıllık ideal point
    ip_path = find_file('un_voting', 'Idealpoint*.csv')
    ip_df = pd.read_csv(ip_path, low_memory=False)
    ip_df.columns = ip_df.columns.str.lower()
    
    print('İdeal point sütunları:', list(ip_df.columns)[:15])
    
    # Sütun isimleri sürüme göre değişebilir — yaygın olanlar:
    # ccode (COW), session/year, idealpoint / ideal_point / idealpointall
    cow_col = next((c for c in ip_df.columns if c in ['ccode', 'cow', 'cowcode']), None)
    year_col = next((c for c in ip_df.columns if c == 'year'), None)
    if not year_col:
        # session var ise session ≈ year - 1945 (yaklaşık)
        session_col = next((c for c in ip_df.columns if 'session' in c), None)
        if session_col:
            ip_df['year'] = ip_df[session_col] + 1945
            year_col = 'year'
    
    ip_col = next((c for c in ip_df.columns if 'idealpoint' in c.replace('_', '')), None)
    
    if not all([cow_col, year_col, ip_col]):
        raise ValueError(f'İdeal point sütunları eksik: {ip_df.columns.tolist()[:10]}')
    
    ip_df = ip_df.rename(columns={cow_col: 'cow_code', year_col: 'year', ip_col: 'ideal'})
    ip_df = ip_df[['cow_code', 'year', 'ideal']].dropna()
    ip_df['cow_code'] = ip_df['cow_code'].astype(int)
    ip_df['year'] = ip_df['year'].astype(int)
    ip_df = ip_df[(ip_df['year'] >= START_YEAR) & (ip_df['year'] <= END_YEAR)]
    ip_df = ip_df.drop_duplicates(['cow_code', 'year'])
    
    log_stage('3c_ideal_read', 'OK', f'ideal point: {ip_col}',
              n_rows=len(ip_df), n_countries=ip_df['cow_code'].nunique())
except Exception as e:
    log_stage('3c_ideal_read', 'FAIL', str(e))
    ip_df = None

In [ ]:
# Her yıl için ülke çiftleri arası mesafe → yakınlık → kenar
# k-NN yaklaşımı: her ülke için en yakın K komşuyu tut (sabit eşik yerine adaptif)
# Böylece farklı yıllardaki ideal point dağılım farklılıkları dengelenir

if ip_df is not None:
    edges_vote_list = []
    K_NEIGHBORS = 30  # her ülke için en yakın 30 komşu (ortalama ~180 ülke × %17)
    
    for year, year_group in tqdm(ip_df.groupby('year'), desc='Oylama mesafeleri'):
        codes = year_group['cow_code'].values
        ideals = year_group['ideal'].values
        n = len(codes)
        
        if n < 2:
            continue
        
        # NxN mesafe matrisi
        dist = np.abs(ideals[:, None] - ideals[None, :])
        
        # Her satır (ülke) için en yakın K komşuyu bul
        k = min(K_NEIGHBORS, n - 1)
        neighbor_set = set()
        for i in range(n):
            # Kendisi hariç en yakın k komşu
            row_dist = dist[i].copy()
            row_dist[i] = np.inf  # kendini çıkar
            nearest_idx = np.argpartition(row_dist, k)[:k]
            for j in nearest_idx:
                # Yönsüz: (min, max) çifti olarak ekle
                ci, cj = int(codes[min(i, j)]), int(codes[max(i, j)])
                if ci != cj:
                    neighbor_set.add((ci, cj, float(dist[i, j])))
        
        for ci, cj, d in neighbor_set:
            edges_vote_list.append({
                'year': year,
                'cow_i': ci,
                'cow_j': cj,
                'weight': float(1.0 / (1.0 + d))  # [0,1] arası yakınlık
            })
    
    edges_vote = pd.DataFrame(edges_vote_list)
    edges_vote = edges_vote.drop_duplicates(['year', 'cow_i', 'cow_j'])
    edges_vote = edges_vote[['year', 'cow_i', 'cow_j', 'weight']]
    
    edges_vote.to_parquet(os.path.join(CANON_DIR, 'edges_vote.parquet'))
    log_stage('3c_vote_edges', 'OK',
              f'k-NN yakınlık (K={K_NEIGHBORS})',
              n_rows=len(edges_vote))

### 3d. Çatışma (COW MID)

MIDB'den her ülkenin her MID'deki tarafı. İki ülke karşıt taraflardaysa o yıl için kenar.

In [ ]:
try:
    midb_path = find_file('cow/cow_mid', 'MIDB*.csv')
    midb = pd.read_csv(midb_path, low_memory=False)
    midb.columns = midb.columns.str.lower()
    
    print(f'MIDB sütunları: {list(midb.columns)}')
    
    # Esnek sütun isimleri
    disp_col = find_col(midb, ['dispnum', 'dispnum3', 'disp_num', 'mid_id'])
    cow_col = find_col(midb, ['ccode', 'cowcode', 'cow_code'])
    sty_col = find_col(midb, ['styear', 'start_year', 'styear_mid', 'startyear'])
    endy_col = find_col(midb, ['endyear', 'end_year', 'endyr'], required=False)
    sidea_col = find_col(midb, ['sidea', 'side_a', 'side'])
    
    # MID süresi içindeki her yıl için kayıt üret
    midb_expanded = []
    for _, r in midb.iterrows():
        try:
            styear = int(r[sty_col])
            endyear = int(r[endy_col]) if endy_col and pd.notna(r[endy_col]) else styear
        except (ValueError, TypeError):
            continue
        for y in range(max(styear, START_YEAR), min(endyear, END_YEAR) + 1):
            midb_expanded.append({
                'dispnum': r[disp_col],
                'cow_code': int(r[cow_col]),
                'year': y,
                'sidea': int(r[sidea_col]),
            })
    midb_exp = pd.DataFrame(midb_expanded)
    
    # MID-yıl içinde karşıt taraflar arası çiftleri üret
    edges_conflict_list = []
    for (dispnum, year), grp in tqdm(midb_exp.groupby(['dispnum', 'year']), desc='MID çiftleri'):
        side_a = grp[grp['sidea'] == 1]['cow_code'].tolist()
        side_b = grp[grp['sidea'] == 0]['cow_code'].tolist()
        for ca in side_a:
            for cb in side_b:
                if ca != cb:
                    edges_conflict_list.append({
                        'year': year,
                        'cow_i': min(ca, cb),
                        'cow_j': max(ca, cb),
                    })
    
    edges_conflict = pd.DataFrame(edges_conflict_list)
    edges_conflict = (
        edges_conflict.groupby(['year', 'cow_i', 'cow_j']).size()
        .reset_index(name='mid_count')
    )
    edges_conflict['weight'] = np.log1p(edges_conflict['mid_count'])
    edges_conflict = edges_conflict[['year', 'cow_i', 'cow_j', 'weight']]
    
    edges_conflict.to_parquet(os.path.join(CANON_DIR, 'edges_conflict.parquet'))
    log_stage('3d_conflict_edges', 'OK',
              f'MID çiftleri, log(1+count)',
              n_rows=len(edges_conflict))
except Exception as e:
    log_stage('3d_conflict_edges', 'FAIL', str(e))

## 4. 5-Yıllık Yumuşatma

Sürekli kenar ağırlıkları (ticaret, oylama) kayan pencereyle yumuşatılır.
Binary kenarlar (ittifak) ve sayım (çatışma) olduğu gibi kalır.

In [ ]:
def smooth_edges(edges_df, window=SMOOTH_WINDOW):
    """Her (cow_i, cow_j) çifti için ağırlığı kayan pencere ortalamasıyla yumuşat."""
    edges_df = edges_df.sort_values(['cow_i', 'cow_j', 'year'])
    edges_df['weight_smooth'] = (
        edges_df.groupby(['cow_i', 'cow_j'])['weight']
        .transform(lambda s: s.rolling(window, center=True, min_periods=1).mean())
    )
    edges_df['weight'] = edges_df['weight_smooth']
    return edges_df.drop(columns=['weight_smooth'])

# Sürekli olanları yumuşat
for fname in ['edges_trade.parquet', 'edges_vote.parquet']:
    fpath = os.path.join(CANON_DIR, fname)
    if os.path.exists(fpath):
        df = pd.read_parquet(fpath)
        df_smooth = smooth_edges(df)
        df_smooth.to_parquet(fpath)
        log_stage(f'4_smooth_{fname}', 'OK', f'{SMOOTH_WINDOW}-yıllık pencere', n_rows=len(df_smooth))

## 5. PyTorch Geometric Snapshot'ları

Her yıl için bir `HeteroData` dosyası: `data/snapshots/graph_{year}.pt`

In [ ]:
import torch
from torch_geometric.data import HeteroData

# Kanonik tabloları belleğe yükle
nodes = pd.read_parquet(os.path.join(CANON_DIR, 'nodes.parquet'))

edge_tables = {}
for rel, fname in [('allies', 'edges_ally.parquet'),
                   ('trades', 'edges_trade.parquet'),
                   ('votes_with', 'edges_vote.parquet'),
                   ('conflicts_with', 'edges_conflict.parquet')]:
    fpath = os.path.join(CANON_DIR, fname)
    if os.path.exists(fpath):
        edge_tables[rel] = pd.read_parquet(fpath)
    else:
        print(f'⚠️  {fname} eksik — bu ilişki tipi atlanacak')

# Düğüm özellik sütunları — sadece sayısal olanlar
feature_cols = [c for c in nodes.columns
                if c not in ['year', 'cow_code', 'state_name']
                and nodes[c].dtype in [np.float32, np.float64, np.int64, np.int32]]
print(f'Düğüm özellik sütunları: {feature_cols}')

In [ ]:
def build_snapshot(year):
    """Verilen yıl için HeteroData oluştur."""
    nodes_y = nodes[nodes['year'] == year].copy()
    if len(nodes_y) == 0:
        return None
    
    # Düğüm index'i: cow_code → 0..N-1 lokal index
    nodes_y = nodes_y.sort_values('cow_code').reset_index(drop=True)
    cow_to_idx = {int(c): i for i, c in enumerate(nodes_y['cow_code'])}
    
    # Düğüm özellik tensörü, eksikleri 0 ile doldur (maske ayrı)
    x_df = nodes_y[feature_cols].copy()
    x_mask = (~x_df.isna()).astype(float).values  # 1 = veri var, 0 = eksik
    x_filled = x_df.fillna(0.0).values
    
    data = HeteroData()
    data['country'].x = torch.tensor(x_filled, dtype=torch.float)
    data['country'].x_mask = torch.tensor(x_mask, dtype=torch.float)
    data['country'].cow_code = torch.tensor(nodes_y['cow_code'].values, dtype=torch.long)
    
    # Her ilişki tipi için kenarlar
    for rel, etable in edge_tables.items():
        etable_y = etable[etable['year'] == year]
        # Sadece bu yılda var olan ülkeler arasındaki kenarlar
        etable_y = etable_y[
            etable_y['cow_i'].isin(cow_to_idx) & etable_y['cow_j'].isin(cow_to_idx)
        ]
        if len(etable_y) == 0:
            continue
        
        src = etable_y['cow_i'].map(cow_to_idx).values
        dst = etable_y['cow_j'].map(cow_to_idx).values
        # Yönsüz → her iki yön
        edge_index = torch.tensor(
            np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])]),
            dtype=torch.long
        )
        edge_weight = torch.tensor(
            np.concatenate([etable_y['weight'].values, etable_y['weight'].values]),
            dtype=torch.float
        )
        data['country', rel, 'country'].edge_index = edge_index
        data['country', rel, 'country'].edge_attr = edge_weight.unsqueeze(1)
    
    data.year = year
    return data

# Test: tek bir yıl üret
test_snap = build_snapshot(1990)
if test_snap:
    print('1990 snapshot örneği:')
    print(test_snap)
    print(f"\nDüğüm sayısı: {test_snap['country'].x.shape[0]}")
    for store in test_snap.edge_stores:
        print(f"  {store._key}: {store.edge_index.shape[1]} kenar")

In [ ]:
# Tüm yıllar için snapshot üret ve diske yaz
n_built = 0
n_failed = 0
failed_years = []

for year in tqdm(range(START_YEAR, END_YEAR + 1), desc='Snapshot üretimi'):
    try:
        snap = build_snapshot(year)
        if snap is None:
            failed_years.append((year, 'düğüm yok'))
            n_failed += 1
            continue
        torch.save(snap, os.path.join(SNAP_DIR, f'graph_{year}.pt'))
        n_built += 1
    except Exception as e:
        failed_years.append((year, str(e)[:60]))
        n_failed += 1

log_stage('5_snapshots', 'OK' if n_failed == 0 else 'WARN',
          f'{n_built} yıl başarılı, {n_failed} başarısız' +
          (f' ({failed_years[:3]}...)' if failed_years else ''),
          n_rows=n_built)

## 6. Doğrulama — Sanity Checks

In [ ]:
# Geçiş yılları sanity check
transition_years = [
    (1990, 'Almanya birleşmesi', 'Almanya sayısı azalmalı'),
    (1991, 'SSCB dağılması', '15 yeni ülke beklenir'),
    (1993, 'Çekoslovakya', '2 yeni ülke'),
    (2011, 'Güney Sudan', '1 yeni ülke'),
]

print('Yıl başına ülke sayısı (kritik dönemler):')
for year in [1946, 1960, 1989, 1990, 1991, 1992, 1993, 2000, 2010, 2011, 2020]:
    n = (nodes['year'] == year).sum()
    print(f'  {year}: {n} ülke')

print('\nGeçiş notları:')
for year, event, expected in transition_years:
    n_before = (nodes['year'] == year - 1).sum()
    n_after = (nodes['year'] == year).sum()
    delta = n_after - n_before
    print(f'  {year} ({event}): Δ = {delta:+d} ülke ({expected})')

In [ ]:
# Snapshot örneği üzerinden istatistik
import torch

sample_year = 1990
snap = torch.load(os.path.join(SNAP_DIR, f'graph_{sample_year}.pt'), weights_only=False)

print(f'=== {sample_year} Snapshot Özeti ===')
print(f"Düğüm sayısı: {snap['country'].x.shape[0]}")
print(f"Düğüm özellik boyutu: {snap['country'].x.shape[1]}")
print(f"Eksiklik oranı (genel): {(1 - snap['country'].x_mask.mean()).item():.1%}")

print('\nKenar tipleri:')
for store in snap.edge_stores:
    n_edges = store.edge_index.shape[1] // 2  # yönsüz olduğu için ikiye böl
    print(f"  {store._key[1]:15s}: {n_edges:>6,} yönsüz kenar")

# NaN / inf kontrolü
print('\nNaN/inf kontrolü:')
for store in snap.node_stores + snap.edge_stores:
    for key, val in store.items():
        if torch.is_tensor(val) and val.dtype.is_floating_point:
            n_nan = torch.isnan(val).sum().item()
            n_inf = torch.isinf(val).sum().item()
            if n_nan > 0 or n_inf > 0:
                print(f'  ⚠️  {store._key} / {key}: {n_nan} NaN, {n_inf} inf')
            else:
                pass
print('  ✅ Tüm tensörler temiz')

## 7. Özet Rapor

In [ ]:
from datetime import datetime

report = pd.DataFrame(stages)
print('='*80)
print('PREPROCESSING ÖZET RAPORU'.center(80))
print(f'{datetime.now().strftime("%Y-%m-%d %H:%M")}'.center(80))
print('='*80)

for status in ['OK', 'WARN', 'FAIL']:
    subset = report[report['status'] == status]
    if len(subset) == 0:
        continue
    icon = {'OK': '✅', 'WARN': '⚠️', 'FAIL': '❌'}[status]
    print(f'\n{icon} {status} ({len(subset)} aşama)')
    print('-'*80)
    for _, r in subset.iterrows():
        suffix = ''
        if pd.notna(r['n_rows']):
            suffix = f" [{int(r['n_rows']):,}"
            if pd.notna(r['n_countries']):
                suffix += f", {int(r['n_countries'])} ülke"
            suffix += ']'
        print(f"  {r['stage']:30s} {r['details']}{suffix}")

print('\n' + '='*80)
n_ok = (report['status'] == 'OK').sum()
n_warn = (report['status'] == 'WARN').sum()
n_fail = (report['status'] == 'FAIL').sum()
print(f'TOPLAM: {n_ok} OK, {n_warn} uyarı, {n_fail} hata')
print('='*80)

# Raporu CSV olarak kaydet
report_path = os.path.join(PROJECT_DIR, f'data/preprocessing_report_{datetime.now().strftime("%Y%m%d_%H%M")}.csv')
report.to_csv(report_path, index=False)
print(f'\n📊 Detaylı rapor: {report_path}')

# Çıktı dosyaları
print('\n📁 Üretilen kanonik tablolar:')
for f in sorted(os.listdir(CANON_DIR)):
    fpath = os.path.join(CANON_DIR, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f'  {f}: {size_mb:.1f} MB')

n_snapshots = len([f for f in os.listdir(SNAP_DIR) if f.startswith('graph_')])
snap_size = sum(os.path.getsize(os.path.join(SNAP_DIR, f))
                for f in os.listdir(SNAP_DIR)) / 1e6
print(f'\n📁 {n_snapshots} graf snapshot\'ı: toplam {snap_size:.1f} MB')

## Sonraki adım

Şu üç şey hazır:
1. `data/canonical/nodes.parquet` — düğüm özellikleri (yıl × ülke × özellikler)
2. `data/canonical/edges_*.parquet` — her ilişki tipi için kenar tabloları
3. `data/snapshots/graph_{year}.pt` — yıllık PyTorch Geometric grafları

**Sonraki notebook (`02b_fix_normalization.ipynb`):**
- Ham (normalize edilmemiş) sütunları drop et
- z-skor normalizasyon (train dönemi istatistikleriyle)
- Snapshot'ları yeniden inşa et

**Ardından (`03_coalition_samples.ipynb`):**
- Tarihsel koalisyon kanonu (NATO, Varşova, AB, vb. — ~50 koalisyon)
- Hedef gözlemler: koalisyon süresi (ATOP), içsel çatışma (MIDB), oylama uyumu (UN ham oylar)
- Sert negatif ve rastgele negatif örnekleyicileri
- Eğitim/val/test bölmesi (1946-1999 / 2000-2009 / 2010-2016)